# 🔗 Bloque 3 — Sesión 1: Concepto de correlación, Pearson y Spearman

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~4 horas

---

### Contenidos de esta sesión
- **3.1** Concepto de correlación: qué mide y qué no mide. Correlación ≠ causalidad
- **3.2** Correlación de Pearson: coeficiente r, interpretación, limitaciones y visualización matricial
- **3.3** Correlación de Spearman: para variables ordinales o distribuciones no normales

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación de la calidad de los datos basadas en cálculos estadísticos para detectar la cobertura y el sesgo.  
> **(CE c)** Se ha documentado la descripción del resultado de las verificaciones sobre el conjunto de datos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 3.1 Concepto de correlación

### ¿Qué mide la correlación?

La **correlación** cuantifica la **fuerza y dirección de la relación lineal** entre dos variables numéricas. Nos dice:
- **Si aumenta una variable, ¿tiende a aumentar o disminuir la otra?** (dirección)
- **¿Con qué regularidad?** (fuerza)

La correlación siempre toma valores entre **−1 y +1**:

| Valor | Interpretación |
|-------|---------------|
| +1 | Relación lineal positiva perfecta |
| +0.7 a +1 | Correlación positiva fuerte |
| +0.3 a +0.7 | Correlación positiva moderada |
| −0.3 a +0.3 | Correlación débil o nula |
| −0.7 a −0.3 | Correlación negativa moderada |
| −1 a −0.7 | Correlación negativa fuerte |
| −1 | Relación lineal negativa perfecta |

In [ ]:
# Galería de correlaciones: distintos valores de r
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
axes = axes.flat

escenarios = [
    (+1.0, 'r = +1.0
(lineal perfecta)'),
    (+0.8, 'r ≈ +0.8
(fuerte positiva)'),
    (+0.4, 'r ≈ +0.4
(moderada positiva)'),
    ( 0.0, 'r ≈ 0.0
(sin relación lineal)'),
    (-0.4, 'r ≈ −0.4
(moderada negativa)'),
    (-0.8, 'r ≈ −0.8
(fuerte negativa)'),
    (-1.0, 'r = −1.0
(lineal perfecta)'),
    ( 0.0, 'r ≈ 0.0
pero relación cuadrática'),
]

x_base = np.linspace(-2, 2, 100)

for ax, (r_obj, titulo) in zip(axes, escenarios):
    if titulo.endswith('cuadrática'):
        x = x_base
        y = x**2 + np.random.normal(0, 0.15, len(x))
    elif abs(r_obj) == 1.0:
        x = x_base
        y = r_obj * x
    else:
        x = x_base
        noise_scale = np.sqrt(1 - r_obj**2)
        y = r_obj * x + np.random.normal(0, noise_scale, len(x))

    r_real, _ = pearsonr(x, y)
    ax.scatter(x, y, s=15, alpha=0.6, color='steelblue')
    ax.set_title(f'{titulo}
(r real = {r_real:.2f})', fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle('Galería de correlaciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Correlación ≠ causalidad

Este es uno de los errores más frecuentes en el análisis de datos. Que dos variables estén correlacionadas **no implica que una cause la otra**. Puede haber:

- **Variable confusora (tercera variable):** una tercera variable causa ambas. *Ejemplo:* el número de salvavidas en una ciudad correlaciona con el número de ahogamientos — ambos dependen del tamaño de la ciudad.
- **Causalidad inversa:** la relación existe pero en sentido contrario al asumido.
- **Correlación espuria:** coincidencia estadística sin relación causal real.

> 💡 **Implicación para ML:** una variable puede ser un buen predictor sin ser causa. Pero si queremos entender el fenómeno o intervenir en él, necesitamos causalidad, no solo correlación.

In [ ]:
# Ejemplo clásico: correlación espuria
# Número de películas de Nicolas Cage y ahogamientos en piscinas (datos reales de Tyler Vigen)
anios = np.arange(1999, 2010)
cage_peliculas = [2, 2, 2, 3, 1, 1, 2, 3, 4, 1, 4]
ahogamientos = [109, 102, 102, 98, 85, 95, 96, 98, 123, 94, 102]

r, p = pearsonr(cage_peliculas, ahogamientos)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

color1, color2 = 'steelblue', 'coral'
ax_l = axes[0]
ax_r = ax_l.twinx()
ax_l.bar(anios - 0.2, cage_peliculas, width=0.4, color=color1, alpha=0.8, label='Películas N. Cage')
ax_r.bar(anios + 0.2, ahogamientos, width=0.4, color=color2, alpha=0.8, label='Ahogamientos')
ax_l.set_ylabel('Nº películas', color=color1)
ax_r.set_ylabel('Ahogamientos en piscinas (EE.UU.)', color=color2)
ax_l.set_title(f'Correlación espuria — r = {r:.3f}, p = {p:.3f}')
ax_l.set_xticks(anios)
ax_l.tick_params(axis='x', rotation=45)

axes[1].scatter(cage_peliculas, ahogamientos, color='steelblue', s=60)
m, b = np.polyfit(cage_peliculas, ahogamientos, 1)
x_fit = np.linspace(min(cage_peliculas), max(cage_peliculas), 100)
axes[1].plot(x_fit, m*x_fit + b, 'crimson', linewidth=2)
axes[1].set_xlabel('Películas de Nicolas Cage ese año')
axes[1].set_ylabel('Ahogamientos en piscinas')
axes[1].set_title('Scatter: aparente correlación positiva')

plt.tight_layout()
plt.show()
print(f"r = {r:.3f}  |  p = {p:.4f}  — correlación estadísticamente significativa... pero espuria.")

---
## 3.2 Correlación de Pearson

La correlación de Pearson mide la **relación lineal** entre dos variables numéricas continuas:

$$r = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

**Supuestos:**
- Ambas variables son numéricas continuas
- La relación entre ellas es lineal (o aproximadamente lineal)
- Idealmente, ambas siguen una distribución normal (o al menos simétrica)
- No hay outliers extremos que distorsionen la estimación

**Limitaciones importantes:**
- Solo detecta relaciones **lineales** (puede valer 0 con relación curvilínea perfecta)
- Muy sensible a **outliers**
- No funciona bien con variables ordinales

In [ ]:
# Pearson sobre el dataset Penguins
peng_num = penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]

r_matrix = peng_num.corr(method='pearson')
print("Matriz de correlación de Pearson (Penguins):")
print(r_matrix.round(3))

In [ ]:
# Heatmap de la matriz de correlación
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(r_matrix, dtype=bool), k=1)  # triángulo superior

sns.heatmap(
    r_matrix, annot=True, fmt='.3f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    ax=ax, square=True
)
ax.set_title('Matriz de correlación de Pearson — Penguins', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix para visualizar todas las relaciones a la vez
g = sns.pairplot(
    penguins, vars=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'],
    hue='species', diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20}
)
g.figure.suptitle('Scatter matrix — Penguins (por especie)', y=1.02, fontsize=13)
plt.show()

### Efecto de los outliers sobre Pearson

In [ ]:
# Demostración: un solo outlier puede cambiar drásticamente r
np.random.seed(1)
x = np.random.normal(0, 1, 50)
y = 0.1 * x + np.random.normal(0, 1, 50)  # relación casi nula

r_sin, _ = pearsonr(x, y)

# Añadimos un solo outlier influyente
x_out = np.append(x, 5.0)
y_out = np.append(y, 5.0)
r_con, _ = pearsonr(x_out, y_out)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(x, y, color='steelblue', s=30, alpha=0.7)
axes[0].set_title(f'Sin outlier
r = {r_sin:.3f}')
axes[0].set_xlim(-4, 6.5)

axes[1].scatter(x, y, color='steelblue', s=30, alpha=0.7, label='Datos')
axes[1].scatter([5.0], [5.0], color='crimson', s=120, zorder=5, label='Outlier', marker='*')
axes[1].set_title(f'Con 1 outlier
r = {r_con:.3f}')
axes[1].set_xlim(-4, 6.5)
axes[1].legend()

plt.suptitle('Sensibilidad de Pearson a outliers', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Sin outlier: r = {r_sin:.3f}  →  Con outlier: r = {r_con:.3f}")
print("Un solo punto cambia la correlación de ~0 a ~0.5")

---
## 3.3 Correlación de Spearman

La correlación de Spearman es una alternativa **no paramétrica** a Pearson. En lugar de trabajar con los valores originales, transforma cada valor en su **rango** (posición ordenada) y calcula la correlación de Pearson sobre esos rangos:

$$r_s = 1 - \frac{6 \sum d_i^2}{n(n^2-1)}$$

donde $d_i$ es la diferencia entre los rangos de $x_i$ e $y_i$.

**Ventajas sobre Pearson:**
- No asume normalidad ni relación lineal: detecta cualquier **relación monótona** (siempre creciente o siempre decreciente)
- Robusta ante outliers (los outliers solo cambian ligeramente su rango)
- Válida para variables **ordinales**

**Cuándo usar Spearman en lugar de Pearson:**
- Distribuciones con sesgo importante o colas pesadas
- Variables ordinales (escala Likert, nivel educativo…)
- Sospecha de relación no lineal pero monótona
- Presencia de outliers que no queremos eliminar

In [ ]:
# Comparación Pearson vs Spearman en distintos escenarios
np.random.seed(42)

escenarios = {
    'Lineal': (lambda x: 0.8*x + np.random.normal(0, 0.5, len(x))),
    'Monótona no lineal (x³)': (lambda x: x**3 + np.random.normal(0, 0.5, len(x))),
    'Con outlier extremo': (lambda x: np.append(0.8*x[:-1] + np.random.normal(0, 0.5, len(x)-1), 15)),
    'No monótona (cuadrática)': (lambda x: x**2 + np.random.normal(0, 0.3, len(x))),
}

x_base = np.linspace(-2, 2, 80)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (titulo, gen_y) in zip(axes.flat, escenarios.items()):
    x = x_base.copy()
    y = gen_y(x)
    if len(y) > len(x): x = np.append(x, 4.0)

    r_p, _ = pearsonr(x, y)
    r_s, _ = spearmanr(x, y)

    ax.scatter(x, y, color='steelblue', s=20, alpha=0.6)
    ax.set_title(f'{titulo}
Pearson r={r_p:.3f}  |  Spearman ρ={r_s:.3f}', fontsize=10)

plt.suptitle('Pearson vs. Spearman en distintos escenarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pearson vs Spearman sobre variables reales con sesgo
fare = titanic['fare'].dropna()
age = titanic['age'].dropna()

# Alineamos índices
df_ps = titanic[['fare', 'age']].dropna()

r_p, p_p = pearsonr(df_ps['fare'], df_ps['age'])
r_s, p_s = spearmanr(df_ps['fare'], df_ps['age'])

print(f"Pearson   r = {r_p:.4f}  (p = {p_p:.4f})")
print(f"Spearman  ρ = {r_s:.4f}  (p = {p_s:.4f})")
print()
print(f"Sesgo de 'fare': {df_ps['fare'].skew():.2f} → distribución muy asimétrica")
print("→ Spearman es más apropiado en este caso por la no normalidad de 'fare'")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(df_ps['age'], df_ps['fare'], alpha=0.3, s=15, color='steelblue')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Tarifa (£)')
axes[0].set_title(f'Escala original
Pearson r={r_p:.3f}  |  Spearman ρ={r_s:.3f}')

axes[1].scatter(df_ps['age'], np.log1p(df_ps['fare']), alpha=0.3, s=15, color='coral')
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('log(Tarifa + 1)')
axes[1].set_title('Escala log — relación más visible')

plt.tight_layout()
plt.show()

In [ ]:
# Función: matriz de correlación con Pearson y Spearman comparadas
def comparar_correlaciones(df, columnas):
    """Muestra heatmaps de Pearson y Spearman lado a lado."""
    datos = df[columnas].dropna()
    r_pearson = datos.corr(method='pearson')
    r_spearman = datos.corr(method='spearman')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, matriz, titulo in zip(axes,
                                   [r_pearson, r_spearman],
                                   ['Pearson (lineal)', 'Spearman (monótona)']):
        sns.heatmap(matriz, annot=True, fmt='.3f', cmap='coolwarm',
                    center=0, vmin=-1, vmax=1,
                    linewidths=0.5, linecolor='white',
                    ax=ax, square=True)
        ax.set_title(titulo, fontsize=12)

    plt.suptitle('Comparación Pearson vs. Spearman', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Diferencias notables
    diff = (r_spearman - r_pearson).abs()
    print("Pares con mayor diferencia |Spearman − Pearson|:")
    diff_triu = diff.where(np.triu(np.ones(diff.shape, dtype=bool), k=1))
    top = diff_triu.stack().sort_values(ascending=False).head(5)
    for (c1, c2), d in top.items():
        print(f"  {c1} vs {c2}: Pearson={r_pearson.loc[c1,c2]:.3f}  Spearman={r_spearman.loc[c1,c2]:.3f}  Δ={d:.3f}")

cols_diamond = ['price', 'carat', 'depth', 'table']
comparar_correlaciones(diamonds, cols_diamond)

---
## 🔧 Ejercicios

### Ejercicio 1 — Identificar correlaciones en Penguins

Calcula la matriz de correlación de Pearson para el dataset `penguins` (solo variables numéricas). Luego:
1. ¿Qué par de variables tiene la correlación más alta? ¿Tiene sentido biológicamente?
2. ¿Hay algún par con correlación negativa? ¿Cómo lo interpretas?
3. Visualiza el par más correlacionado con un scatter coloreado por especie.

In [ ]:
# 1. Matriz de correlación de Pearson
# cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
# r_matrix = penguins[cols].corr()
# print(r_matrix.round(3))

# 2. Par con correlación más alta (excluye diagonal)
# ...

# 3. Scatter del par más correlacionado, coloreado por especie
# sns.scatterplot(data=penguins, x=..., y=..., hue='species')
# ...


### Ejercicio 2 — Pearson vs. Spearman en Diamonds

Aplica `comparar_correlaciones()` al dataset `diamonds` con las variables `price`, `carat`, `depth` y `table`. ¿En qué pares difieren más los dos coeficientes? ¿Por qué crees que ocurre eso (pista: revisa el sesgo de esas variables)?

In [ ]:
# comparar_correlaciones(diamonds, ['price', 'carat', 'depth', 'table'])

# ¿Qué par tiene la mayor diferencia entre Pearson y Spearman?
# Revisa el sesgo de las variables implicadas:
# diamonds[['price', 'carat', 'depth', 'table']].skew()


### Ejercicio 3 — Correlación espuria

Busca en el dataset `titanic` una correlación que parezca sorprendente. Por ejemplo, ¿existe relación entre el número de hermanos/cónyuge a bordo (`sibsp`) y la tarifa pagada (`fare`)? ¿Y entre `parch` y `fare`?

Calcula los coeficientes de Pearson y Spearman, visualiza el scatter y razona: ¿es causal, confusora o espuria?

In [ ]:
# df_t = titanic[['sibsp', 'parch', 'fare', 'pclass']].dropna()
# comparar_correlaciones(df_t, ['sibsp', 'parch', 'fare', 'pclass'])

# Scatter sibsp vs fare coloreado por pclass
# sns.scatterplot(data=titanic, x='sibsp', y='fare', hue='pclass', alpha=0.5)

# ¿Qué variable de confusión podría explicar la relación?


---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Correlación** | Mide fuerza y dirección de la relación entre dos variables. Rango [−1, +1]. |
| **Correlación ≠ causalidad** | Correlación alta no implica que una variable cause la otra. |
| **Pearson (r)** | Para relaciones lineales entre variables continuas. Sensible a outliers y no normalidad. |
| **Spearman (ρ)** | Para relaciones monótonas, variables ordinales o distribuciones asimétricas. Robusto. |
| **Heatmap** | Visualización eficiente de la matriz de correlación. Identifica grupos de variables relacionadas. |
| **Scatter matrix** | Muestra todas las relaciones bivariadas a la vez. Útil con 4-8 variables. |
| **Cuándo usar Spearman** | Distribución asimétrica, outliers, variable ordinal o relación no lineal monótona. |

### 🔜 Próxima sesión
**Bloque 3 — Sesión 2:** Cramér's V para variables categóricas, p-valor e intervalos de confianza, y aplicación a la selección de atributos para el modelado.